# 04 — Entrenamiento del Modelo

Entrena **MarketValueNet** (Wide & Deep) sobre los datos procesados en `data/processed/`.

**Prerequisitos:**
```bash
python -m src.features.pipeline          # genera data/processed/*.npy
# o con embeddings dummy (rápido, para probar):
python -m src.features.pipeline --dummy-text
```

**Secciones:**
1. Cargar dataset y estadísticas
2. Split temporal (train/val)
3. Arquitectura del modelo
4. Entrenamiento
5. Curvas de aprendizaje
6. Evaluación en validación
7. Guardar modelo

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

PROCESSED = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'data' / 'models'

required = ['numerical_features.npy', 'labels.npy', 'category_ids.npy', 'text_embeddings.npy']
missing = [f for f in required if not (PROCESSED / f).exists()]
assert not missing, (
    f"Faltan archivos en data/processed/: {missing}\n"
    "Correr primero: python -m src.features.pipeline"
)
print("Datos procesados disponibles:")
for f in required + ['end_dates.npy', 'pipeline/scaler.pkl']:
    exists = (PROCESSED / f).exists()
    print(f"  {'✓' if exists else '✗'} {f}")

## 1. Cargar dataset y estadísticas

In [ ]:
from src.model.dataset import PolymarketDataset, create_dataloaders
from src.features.numerical import NUMERICAL_FEATURE_NAMES, NUM_NUMERICAL_FEATURES

dataset = PolymarketDataset.from_numpy_dir(str(PROCESSED))

n = len(dataset)
n_pos = int(dataset.labels.sum().item())
n_neg = n - n_pos

print(f"Dataset: {n:,} mercados")
print(f"  Positivos (Buy):   {n_pos:,} ({100*n_pos/n:.1f}%)")
print(f"  Negativos (No Buy): {n_neg:,} ({100*n_neg/n:.1f}%)")
print(f"  Features numéricas: {dataset.numerical.shape[1]}")
print(f"  Text embed dim:     {dataset.text_emb.shape[1]}")
print(f"  Split temporal:     {'sí' if dataset.timestamps is not None else 'no (random split)'}")

## 2. Split temporal train / val

Los mercados más antiguos van a train, los más recientes a val.
Esto evita **data leakage temporal**: el modelo no puede ver el futuro durante entrenamiento.

In [ ]:
import pandas as pd

VAL_SPLIT = 0.2
BATCH_SIZE = 64

train_loader, val_loader = create_dataloaders(
    dataset, batch_size=BATCH_SIZE, val_split=VAL_SPLIT,
    temporal_split=True, use_weighted_sampler=True,
)

n_val = int(n * VAL_SPLIT)
n_train = n - n_val
print(f"Train: {n_train:,} | Val: {n_val:,}")

if dataset.timestamps is not None:
    import datetime
    ts = dataset.timestamps
    sorted_ts = np.sort(ts)
    cutoff_ts = sorted_ts[n_train]
    cutoff_date = datetime.datetime.fromtimestamp(cutoff_ts, tz=datetime.timezone.utc)
    print(f"Corte temporal: {cutoff_date.strftime('%Y-%m-%d')}")
    print(f"  Train: mercados anteriores al corte")
    print(f"  Val:   mercados posteriores al corte")

    fig, ax = plt.subplots(figsize=(10, 3))
    dates = [datetime.datetime.fromtimestamp(t, tz=datetime.timezone.utc) for t in sorted_ts]
    ax.scatter(dates[:n_train], [0]*n_train, alpha=0.3, s=5, color=PALETTE[0], label=f'Train ({n_train:,})')
    ax.scatter(dates[n_train:], [0]*n_val, alpha=0.3, s=5, color=PALETTE[1], label=f'Val ({n_val:,})')
    ax.axvline(cutoff_date, color='black', linestyle='--', lw=1.5, label='Corte')
    ax.set_title('Split temporal de mercados')
    ax.set_yticks([])
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'figures' / 'train_temporal_split.png', bbox_inches='tight')
    plt.show()

## 3. Arquitectura del modelo

**Wide & Deep**: dos caminos que se combinan antes del head de clasificación.

- **Wide**: `numerical(23) → Linear(23, 32) → ReLU` — aprende relaciones lineales directas
- **Deep**: `[numerical + cat_emb(8) + text(384)] → BN+ReLU+Dropout × [256, 128, 64]` — aprende interacciones no lineales
- **Head**: `Linear → Sigmoid` → score ∈ (0, 1)

In [ ]:
from src.model.architecture import MarketValueNet

model = MarketValueNet(
    num_numerical_features=dataset.numerical.shape[1],
    num_categories=10,
    category_embed_dim=8,
    text_embed_dim=dataset.text_emb.shape[1],
    hidden_dims=[256, 128, 64],
    dropout=0.3,
    task="classification",
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nParámetros totales:     {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Dispositivo: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 4. Entrenamiento

- **Optimizador**: AdamW con weight decay 1e-4
- **Scheduler**: CosineAnnealing (lr decae suavemente hasta 0)
- **Criterio de parada**: mejor val AUC-ROC (no val loss)
- **Balance de clases**: WeightedRandomSampler en train (oversampling de positivos)

El modelo se guarda en `data/models/best_market_model.pt`.

In [ ]:
from src.model.train import train_model

EPOCHS = 50

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=1e-3,
    save_dir=str(MODELS_DIR),
)

## 5. Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

ax = axes[0]
ax.plot(epochs_range, history['train_loss'], color=PALETTE[0], label='Train loss')
ax.plot(epochs_range, history['val_loss'], color=PALETTE[1], label='Val loss')
ax.set_title('Loss por época')
ax.set_xlabel('Época')
ax.set_ylabel('BCE Loss')
ax.legend()

ax = axes[1]
ax.plot(epochs_range, history['val_auc'], color=PALETTE[2], label='Val AUC-ROC', linewidth=2)
ax.plot(epochs_range, history['val_accuracy'], color=PALETTE[3], label='Val Accuracy', linestyle='--')
ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='Random baseline')
best_epoch = int(np.argmax(history['val_auc'])) + 1
best_auc = max(history['val_auc'])
ax.axvline(best_epoch, color='black', linestyle='--', lw=1, label=f'Best epoch ({best_epoch})')
ax.set_title(f'Val AUC-ROC | Mejor: {best_auc:.4f} en época {best_epoch}')
ax.set_xlabel('Época')
ax.legend()
ax.set_ylim(0.45, 1.0)

plt.suptitle('Curvas de aprendizaje — MarketValueNet', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'train_learning_curves.png', bbox_inches='tight')
plt.show()
print(f"Mejor val AUC: {best_auc:.4f} (época {best_epoch})")
print(f"Val AUC final: {history['val_auc'][-1]:.4f}")

## 6. Evaluación en validación

Cargamos el mejor checkpoint (guardado por AUC) y evaluamos en el set de validación.

In [ ]:
from src.model.evaluate import evaluate_model, print_evaluation

# Cargar mejor checkpoint
best_model = MarketValueNet(
    num_numerical_features=dataset.numerical.shape[1],
    num_categories=10,
    category_embed_dim=8,
    text_embed_dim=dataset.text_emb.shape[1],
    hidden_dims=[256, 128, 64],
    dropout=0.3,
    task="classification",
)
best_model.load_state_dict(
    torch.load(MODELS_DIR / 'best_market_model.pt', weights_only=True)
)

results = evaluate_model(best_model, val_loader)
print_evaluation(results)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Confusion matrix
ax = axes[0]
cm = results['confusion_matrix']
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Buy', 'Buy'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix (umbral 0.5)')

# Score distribution
ax = axes[1]
scores = results['scores']
labels_arr = results['labels']
for val, label, color in [(1, 'Buy (1)', PALETTE[2]), (0, 'No Buy (0)', PALETTE[3])]:
    ax.hist(scores[labels_arr == val], bins=40, alpha=0.6, color=color,
            label=f'{label}', density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Umbral 0.5')
ax.axvline(0.6, color='gray', linestyle=':', lw=1.5, label='Umbral 0.6')
ax.set_title('Distribución de scores por clase')
ax.set_xlabel('Model score')
ax.legend(fontsize=9)

# ROC curve
ax = axes[2]
fpr, tpr, _ = roc_curve(labels_arr, scores)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color=PALETTE[0], lw=2, label=f'AUC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()

plt.suptitle('Evaluación en validación — MarketValueNet', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'train_evaluation.png', bbox_inches='tight')
plt.show()

## 7. Análisis de umbrales

El umbral óptimo no siempre es 0.5. Evaluamos precision, recall y F1 a distintos umbrales
para elegir uno que maximice el objetivo: **precisión alta con recall razonable**
(preferimos pocas señales buenas sobre muchas señales ruidosas).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = np.arange(0.3, 0.9, 0.05)
rows = []
for t in thresholds:
    preds = (scores >= t).astype(float)
    if preds.sum() == 0:
        continue
    rows.append({
        'threshold': round(t, 2),
        'precision': precision_score(labels_arr, preds, zero_division=0),
        'recall': recall_score(labels_arr, preds, zero_division=0),
        'f1': f1_score(labels_arr, preds, zero_division=0),
        'n_buy_signals': int(preds.sum()),
        'pct_signals': 100 * preds.mean(),
    })

df_thresh = pd.DataFrame(rows)
print(df_thresh.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh['threshold'], df_thresh['precision'], marker='o', color=PALETTE[2], label='Precision')
ax.plot(df_thresh['threshold'], df_thresh['recall'], marker='s', color=PALETTE[3], label='Recall')
ax.plot(df_thresh['threshold'], df_thresh['f1'], marker='^', color=PALETTE[0], label='F1')
ax.axvline(0.6, color='gray', linestyle='--', lw=1.5, label='Umbral config (0.6)')
ax.axvline(0.75, color='black', linestyle=':', lw=1.5, label='Strong buy (0.75)')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Métrica')
ax.set_title('Precision / Recall / F1 por umbral')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'train_threshold_analysis.png', bbox_inches='tight')
plt.show()